# 11 — Rigorous TF-IDF + SVM audit

Dedicated stress test of the non-neural floor (`tfidf-svm`). Goal: show the
near-BERT accuracy result is reproducible, characterize failure modes, and
document ablations so the default config is not cherry-picked.

Uses the same splits / `predict_toxic_proba` contract as the fair harness.
Set `QUICK_MODE = False` for submission-quality sizes.


In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

QUICK_MODE = True
THRESHOLD = 0.5
EVAL_N = 200 if QUICK_MODE else 500
TRAIN_CAP = 20_000 if QUICK_MODE else None  # None = full train
ABLATION_TRAIN_N = 8_000 if QUICK_MODE else 20_000
ROBUST_N = 80 if QUICK_MODE else 200

_src = None
for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_p / "src" / "paths.py").is_file():
        _src = str(_p / "src")
        break
if _src is None:
    raise FileNotFoundError("Cannot find DeepProjectHate/src/paths.py")
if _src not in sys.path:
    sys.path.insert(0, _src)

for _m in ("paths", "baselines", "splits", "fairness", "perturb", "intervention"):
    sys.modules.pop(_m, None)

from baselines import TfidfSvmScorer
from fairness import CONTROL_TEXTS, build_identity_probe_df, control_fpr, per_group_fpr
from intervention import mask_terms
from paths import PROCESSED_DIR, PROJECT_ROOT, SEED
from perturb import PERTURBATIONS
from splits import get_split_frame, sample_split_frame

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("ROOT", PROJECT_ROOT)
print(f"QUICK_MODE={QUICK_MODE} EVAL_N={EVAL_N} TRAIN_CAP={TRAIN_CAP}")


## 1. Fit default SVM on train split


In [ ]:
train = get_split_frame("train")
if TRAIN_CAP is not None and len(train) > TRAIN_CAP:
    train = train.sample(TRAIN_CAP, random_state=SEED).reset_index(drop=True)

val = sample_split_frame("val", n=EVAL_N, seed=SEED)
test = sample_split_frame("test", n=EVAL_N, seed=SEED)

y_train = train["toxic"].astype(int).values
texts_train = train["comment_text"].astype(str).tolist()

svm = TfidfSvmScorer()  # default: 50k, (1,2)-grams, calibrated LinearSVC
t0 = time.perf_counter()
svm.fit(texts_train, y_train)
fit_s = time.perf_counter() - t0
print(f"fit: {fit_s:.1f}s | vocab={svm.n_params:,} | train_n={len(train):,}")
print(f"val_n={len(val)} test_n={len(test)}")


## 2. Held-out metrics (val + test)


In [ ]:
def binary_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }
    if len(np.unique(y_true)) > 1:
        out["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    else:
        out["roc_auc"] = float("nan")
        out["pr_auc"] = float("nan")
    return out


def score_split(df, scorer):
    texts = df["comment_text"].astype(str).tolist()
    y = df["toxic"].astype(int).values
    t0 = time.perf_counter()
    p = np.asarray(scorer.predict_toxic_proba(texts), dtype=float)
    ms = 1000.0 * (time.perf_counter() - t0) / max(len(texts), 1)
    m = binary_metrics(y, p)
    m["ms_per_comment"] = round(ms, 3)
    return p, y, m


val_p, val_y, val_m = score_split(val, svm)
test_p, test_y, test_m = score_split(test, svm)
metrics_df = pd.DataFrame([{"split": "val", **val_m}, {"split": "test", **test_m}])
display(metrics_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cm = confusion_matrix(test_y, (test_p >= THRESHOLD).astype(int))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["pred0", "pred1"], yticklabels=["gold0", "gold1"])
axes[0].set_title("Test confusion (thr=0.5)")

if len(np.unique(test_y)) > 1:
    fpr, tpr, _ = roc_curve(test_y, test_p)
    axes[1].plot(fpr, tpr, label=f"ROC AUC={test_m['roc_auc']:.3f}")
    axes[1].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].legend()
    axes[1].set_title("Test ROC")
plt.tight_layout(); plt.show()


## 3. Threshold sweep (val → apply on test)


In [ ]:
ths = np.linspace(0.05, 0.95, 19)
sweep = []
for thr in ths:
    sweep.append({"threshold": thr, "val_f1": binary_metrics(val_y, val_p, thr)["f1"]})
sweep_df = pd.DataFrame(sweep)
best_thr = float(sweep_df.loc[sweep_df["val_f1"].idxmax(), "threshold"])
test_at_best = binary_metrics(test_y, test_p, best_thr)
test_at_05 = binary_metrics(test_y, test_p, 0.5)

print(f"best val threshold={best_thr:.2f}")
print("test @0.50:", {k: round(v, 3) for k, v in test_at_05.items()})
print("test @best:", {k: round(v, 3) for k, v in test_at_best.items()})

ax = sns.lineplot(data=sweep_df, x="threshold", y="val_f1", marker="o")
ax.axvline(0.5, color="gray", ls="--", label="0.5")
ax.axvline(best_thr, color="crimson", ls="--", label=f"best={best_thr:.2f}")
ax.set_title("Val F1 vs threshold"); ax.legend()
plt.tight_layout(); plt.show()


## 4. Ablations (same val/test slices)

Small grid over n-grams, `max_features`, and stop-words. Trains on a capped
train subset for speed; compare on the **same** test slice as §2.


In [ ]:
abl_train = train
if len(abl_train) > ABLATION_TRAIN_N:
    abl_train = abl_train.sample(ABLATION_TRAIN_N, random_state=SEED).reset_index(drop=True)
abl_texts = abl_train["comment_text"].astype(str).tolist()
abl_y = abl_train["toxic"].astype(int).values

configs = [
    {"ngram_range": (1, 1), "max_features": 50_000, "stop_words": None},
    {"ngram_range": (1, 2), "max_features": 10_000, "stop_words": None},
    {"ngram_range": (1, 2), "max_features": 50_000, "stop_words": None},  # default-ish
    {"ngram_range": (1, 2), "max_features": 100_000, "stop_words": None},
    {"ngram_range": (1, 3), "max_features": 50_000, "stop_words": None},
    {"ngram_range": (1, 2), "max_features": 50_000, "stop_words": "english"},
]

abl_rows = []
for cfg in configs:
    scorer = TfidfSvmScorer(max_features=cfg["max_features"], ngram_range=cfg["ngram_range"])
    if cfg["stop_words"] is not None:
        scorer.pipeline.named_steps["tfidf"].set_params(stop_words=cfg["stop_words"])
    t0 = time.perf_counter()
    scorer.fit(abl_texts, abl_y)
    fit_t = time.perf_counter() - t0
    p, y, m = score_split(test, scorer)
    abl_rows.append({
        "ngram": str(cfg["ngram_range"]),
        "max_features": cfg["max_features"],
        "stop_words": cfg["stop_words"] or "none",
        "vocab": scorer.n_params,
        "fit_s": round(fit_t, 1),
        "test_f1": m["f1"],
        "test_pr_auc": m["pr_auc"],
        "ms_per_comment": m["ms_per_comment"],
    })
abl_df = pd.DataFrame(abl_rows).sort_values("test_f1", ascending=False)
display(abl_df.round(3))

ax = sns.barplot(data=abl_df, x="ngram", y="test_f1", hue="max_features")
ax.set_title("Ablation: test F1"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()


## 5. Error analysis (FP / FN)


In [ ]:
err = test.copy()
err["prob"] = test_p
err["pred"] = (test_p >= THRESHOLD).astype(int)
err["gold"] = test_y
fp = err[(err["gold"] == 0) & (err["pred"] == 1)].sort_values("prob", ascending=False).head(20)
fn = err[(err["gold"] == 1) & (err["pred"] == 0)].sort_values("prob").head(20)

print(f"FP count={( (err.gold==0)&(err.pred==1) ).sum()}  FN count={( (err.gold==1)&(err.pred==0) ).sum()}")
print("\\n--- False positives (top P) ---")
display(fp[["comment_text", "prob"]].assign(comment_text=lambda d: d["comment_text"].str.slice(0, 180)))
print("--- False negatives (lowest P) ---")
display(fn[["comment_text", "prob"]].assign(comment_text=lambda d: d["comment_text"].str.slice(0, 180)))

plot_df = err.assign(
    n_chars=err["comment_text"].astype(str).str.len(),
    label=err["gold"].map({0: "non-toxic", 1: "toxic"}),
)
sns.scatterplot(data=plot_df, x="n_chars", y="prob", hue="label", alpha=0.5)
plt.axhline(THRESHOLD, color="gray", ls="--")
plt.title("Length vs P(toxic)"); plt.tight_layout(); plt.show()


## 6. Fairness probe (identity FPR)


In [ ]:
probe = build_identity_probe_df()
probe_p = svm.predict_toxic_proba(probe["text"].tolist())
group_fpr = per_group_fpr(probe, probe_p, threshold=THRESHOLD)
ctrl = control_fpr(svm.predict_toxic_proba(CONTROL_TEXTS), threshold=THRESHOLD)
overall_fpr = float(group_fpr["fpr"].mean())
print(f"overall identity FPR={overall_fpr:.3f} | control FPR={ctrl['fpr']:.3f}")
display(group_fpr.sort_values("fpr", ascending=False).head(10).round(3))

ax = sns.barplot(
    data=group_fpr.sort_values("fpr", ascending=False).head(12),
    x="group", y="fpr", color="crimson",
)
ax.set_title("SVM: worst identity groups (FPR)"); ax.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()


## 7. Robustness (evasion on toxic comments)


In [ ]:
toxic_pool = get_split_frame("test")
toxic_pool = toxic_pool[toxic_pool["toxic"] == 1]
if len(toxic_pool) > ROBUST_N:
    toxic_pool = toxic_pool.sample(ROBUST_N, random_state=SEED)
base_texts = toxic_pool["comment_text"].astype(str).tolist()
base_p = svm.predict_toxic_proba(base_texts)
base_flag = (base_p >= THRESHOLD)
base_rate = float(base_flag.mean())
print(f"baseline detection rate on toxic sample: {base_rate:.3f} (n={len(base_texts)})")

rob_rows = []
for name, fn in PERTURBATIONS.items():
    pert = [fn(t) for t in base_texts]
    pp = svm.predict_toxic_proba(pert)
    still = (pp >= THRESHOLD)
    # evasion among those originally flagged
    flagged_idx = np.where(base_flag)[0]
    if len(flagged_idx):
        evasion = float((~still[flagged_idx]).mean())
    else:
        evasion = float("nan")
    rob_rows.append({
        "perturbation": name,
        "detection_rate": float(still.mean()),
        "evasion_rate": evasion,
    })
rob_df = pd.DataFrame(rob_rows).sort_values("evasion_rate", ascending=False)
display(rob_df.round(3))

ax = sns.barplot(data=rob_df, x="perturbation", y="evasion_rate", color="darkorange")
ax.set_title("SVM evasion rate"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()


## 8. Inference masking (identity token shortcut)


In [ ]:
probe_masked = probe.copy()
probe_masked["text_masked"] = probe_masked["text"].map(mask_terms)
p_id = svm.predict_toxic_proba(probe["text"].tolist())
p_mask = svm.predict_toxic_proba(probe_masked["text_masked"].tolist())
fpr_id = float((p_id >= THRESHOLD).mean())
fpr_mask = float((p_mask >= THRESHOLD).mean())
print(f"FPR with identity={fpr_id:.3f} | FPR masked={fpr_mask:.3f} | drop={fpr_id - fpr_mask:.3f}")
print(f"mean P with identity={p_id.mean():.3f} | masked={p_mask.mean():.3f}")

mask_plot = pd.DataFrame({
    "stage": ["with_identity", "masked"],
    "fpr": [fpr_id, fpr_mask],
})
sns.barplot(data=mask_plot, x="stage", y="fpr", color="steelblue")
plt.title("SVM inference masking"); plt.ylim(0, max(0.2, fpr_id * 1.2))
plt.tight_layout(); plt.show()


## 9. Cache-first vs toxic-bert (no forced re-run)


In [ ]:
comp_path = PROCESSED_DIR / "model_comparison.json"
fair_path = PROCESSED_DIR / "fairness_results.json"
rows = []
if comp_path.exists():
    payload = json.loads(comp_path.read_text(encoding="utf-8"))
    for key in ("tfidf-svm", "toxic-bert"):
        block = payload.get("models", {}).get(key)
        if not block:
            print(f"[CACHE MISS] comparison/{key}")
            continue
        t = block.get("splits", {}).get("test", {})
        rows.append({
            "model": key,
            "source": "model_comparison.json",
            "test_f1": t.get("f1"),
            "test_pr_auc": t.get("pr_auc"),
            "ms_per_comment": t.get("ms_per_comment"),
            "n_params": block.get("n_params"),
        })
else:
    print("No", comp_path)

# Always include this notebook's live SVM test metrics
rows.append({
    "model": "tfidf-svm (this notebook)",
    "source": "live",
    "test_f1": test_m["f1"],
    "test_pr_auc": test_m["pr_auc"],
    "ms_per_comment": test_m["ms_per_comment"],
    "n_params": svm.n_params,
})
cmp_df = pd.DataFrame(rows)
display(cmp_df.round(3))

if fair_path.exists():
    fair = json.loads(fair_path.read_text(encoding="utf-8"))
    frows = []
    for key in ("tfidf-svm", "toxic-bert"):
        b = fair.get("models", {}).get(key)
        if b:
            frows.append({"model": key, "overall_fpr": b.get("overall_fpr"), "control_fpr": b.get("control_fpr")})
    frows.append({"model": "tfidf-svm (this notebook)", "overall_fpr": overall_fpr, "control_fpr": ctrl["fpr"]})
    display(pd.DataFrame(frows).round(3))
else:
    print("No", fair_path)


## 10. Save audit JSON


In [ ]:
out = {
    "seed": SEED,
    "quick_mode": QUICK_MODE,
    "eval_n": EVAL_N,
    "train_n": len(train),
    "threshold": THRESHOLD,
    "best_val_threshold": best_thr,
    "fit_seconds": fit_s,
    "n_params": svm.n_params,
    "metrics": {"val": val_m, "test": test_m, "test_at_best_thr": test_at_best},
    "ablations": abl_df.to_dict(orient="records"),
    "fairness": {
        "overall_fpr": overall_fpr,
        "control_fpr": ctrl["fpr"],
        "worst_groups": group_fpr.sort_values("fpr", ascending=False).head(5).to_dict(orient="records"),
    },
    "robustness": rob_df.to_dict(orient="records"),
    "masking": {
        "fpr_with_identity": fpr_id,
        "fpr_masked": fpr_mask,
        "mean_prob_with_identity": float(p_id.mean()),
        "mean_prob_masked": float(p_mask.mean()),
    },
}
path = PROCESSED_DIR / "svm_rigorous_audit.json"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2), encoding="utf-8")
print("Wrote", path)
